In [ ]:
"""
Bronze → Silver ETL Pipeline
==============================
Extracts raw billing transaction JSON from the bronze layer,
flattens / transforms it, and loads into 8 silver layer tables:

    silver.sales              — transaction header fact table
    silver.sales_items        — line-item fact table (exploded from items[])
    silver.customers          — customer dimension (deduplicated)
    silver.stores             — store dimension (deduplicated)
    silver.billing_counters   — counter dimension (deduplicated)
    silver.payments           — payment fact table
    silver.returns            — return fact table (only rows with return data)
    silver.stock_movements    — inventory movement fact table

Load modes:
    "full"        → truncate all silver tables, then reload from scratch
    "incremental" → append only, no truncation

Usage (import):
    from bronze_to_silver import main, DBConfig
    main(source_db, dest_db, start_date="2024-01-01",
         end_date="2024-01-31", load_type="incremental")
"""

from __future__ import annotations

import json
import logging
import sys
from dataclasses import dataclass
from typing import Optional

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine


# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)


# ============================================================
# CONFIG
# ============================================================
@dataclass
class DBConfig:
    """Holds all database connection parameters."""
    host:     str
    db:       str
    user:     str
    password: str
    port:     int = 5432

    def to_uri(self) -> str:
        return (
            f"postgresql://{self.user}:{self.password}"
            f"@{self.host}:{self.port}/{self.db}"
        )


# ============================================================
# CONSTANTS
# ============================================================

BRONZE_TABLE = "bronze.billing_transactions"

# All silver destination tables — order matters for CASCADE truncation
SILVER_TABLES = [
    "sales",
    "sales_items",
    "customers",
    "stores",
    "billing_counters",
    "payments",
    "returns",
    "stock_movements",
]

# JSON string columns to deserialize before transforming
JSON_COLUMNS = [
    "store",
    "billing_counter",
    "customer",
    "items",
    "payment",
    "billing_summary",
    "data_mart",
    "return_info",
]

# Keys to extract from the billing_summary nested object
BILLING_SUMMARY_FIELDS = [
    "total_line_items",
    "total_quantity",
    "subtotal",
    "total_txn_discount",
    "tax_amount",
    "gross_amount",
    "loyalty_discount",
    "final_amount",
    "amount_paid",
    "change_returned",
    "currency",
]


# ============================================================
# DATABASE
# ============================================================
def get_engine(config: DBConfig) -> Engine:
    """Build and return a SQLAlchemy engine from a DBConfig."""
    logger.info(
        f"Connecting → {config.user}@{config.host}:{config.port}/{config.db}"
    )
    return create_engine(config.to_uri())


# ============================================================
# EXTRACT
# ============================================================
def extract_bronze(
    engine:     Engine,
    start_date: Optional[str] = None,
    end_date:   Optional[str] = None,
) -> pd.DataFrame:
    """
    Read raw records from bronze.billing_transactions.

    Args:
        engine:     Source DB engine.
        start_date: Inclusive lower bound on transaction_date (YYYY-MM-DD).
        end_date:   Inclusive upper bound on transaction_date (YYYY-MM-DD).

    Returns:
        Raw DataFrame — one row per transaction.
    """
    base = f"SELECT * FROM {BRONZE_TABLE}"

    if start_date and end_date:
        where = f" WHERE transaction_date BETWEEN '{start_date}' AND '{end_date}'"
    elif start_date:
        where = f" WHERE transaction_date = '{start_date}'"
    else:
        where = ""

    query = base + where
    logger.info(f"Extracting bronze | {query}")

    df = pd.read_sql(query, engine)
    logger.info(f"Extracted {len(df)} records from bronze layer")
    return df


# ============================================================
# PARSE
# ============================================================
def parse_json_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deserialize JSON string columns into Python dicts / lists.

    - Columns absent from the DataFrame are skipped with a warning.
    - Null values are preserved as None.
    - Already-parsed objects (dict / list) are left unchanged.
    """
    df = df.copy()

    present = [c for c in JSON_COLUMNS if c in df.columns]
    missing = set(JSON_COLUMNS) - set(present)

    if missing:
        logger.warning(f"JSON columns not found in data (skipped): {sorted(missing)}")

    for col in present:
        df[col] = df[col].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )

    logger.info(f"Parsed {len(present)} JSON columns")
    return df


# ============================================================
# HELPERS
# ============================================================
def _safe_get(series: pd.Series, key: str) -> pd.Series:
    """
    Null-safe key extractor from a Series of dicts.
    Returns None for any non-dict element instead of raising KeyError.
    """
    return series.apply(lambda x: x.get(key) if isinstance(x, dict) else None)


# ============================================================
# TRANSFORMS
# ============================================================

# ── 1. silver.sales  ─────────────────────────────────────────
def transform_sales(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten transaction header → silver.sales.
    One row per transaction.
    """
    logger.info("Transforming → silver.sales")

    sales_df = pd.DataFrame({
        # identifiers
        "transaction_id":        df["transaction_id"],
        "invoice_number":        df["invoice_number"],
        "order_status":          df["order_status"],

        # timestamps & date parts
        "transaction_timestamp": df["transaction_timestamp"],
        "transaction_date":      df["transaction_date"],
        "transaction_time":      df["transaction_time"],
        "payment_timestamp":     df["payment_timestamp"],
        "day_of_week":           df["day_of_week"],
        "week_number":           df["week_number"],
        "month":                 df["month"],
        "quarter":               df["quarter"],
        "fiscal_year":           df["fiscal_year"],

        # foreign keys extracted from nested JSON
        "store_id":              _safe_get(df["store"],           "store_id"),
        "counter_id":            _safe_get(df["billing_counter"], "counter_id"),
        "customer_id":           _safe_get(df["customer"],        "customer_id"),

        # channel
        "sales_channel":         df["sales_channel"],

        # billing summary fields (driven by constant — add fields there, not here)
        **{
            field: _safe_get(df["billing_summary"], field)
            for field in BILLING_SUMMARY_FIELDS
        },
    })

    logger.info(f"silver.sales → {len(sales_df)} rows")
    return sales_df


# ── 2. silver.sales_items  ───────────────────────────────────
def transform_sales_items(df: pd.DataFrame) -> pd.DataFrame:
    """
    Explode items[] array → silver.sales_items.
    One row per line item per transaction.
    transaction_id is injected into every item row.
    """
    logger.info("Transforming → silver.sales_items")

    rows = []
    for _, row in df.iterrows():
        items = row.get("items")
        if not isinstance(items, list):
            continue
        for item in items:
            if isinstance(item, dict):
                rows.append({**item, "transaction_id": row["transaction_id"]})

    items_df = pd.DataFrame(rows)
    logger.info(f"silver.sales_items → {len(items_df)} rows")
    return items_df


# ── 3. silver.customers  ─────────────────────────────────────
def transform_customers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten customer JSON → silver.customers.
    Deduplicated by customer_id (last occurrence kept — most recent data wins).
    """
    logger.info("Transforming → silver.customers")

    customers_df = (
        pd.json_normalize(df["customer"])
        .drop_duplicates(subset=["customer_id"], keep="last")
        .reset_index(drop=True)
    )

    logger.info(f"silver.customers → {len(customers_df)} rows")
    return customers_df


# ── 4. silver.stores  ────────────────────────────────────────
def transform_stores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten store JSON → silver.stores.
    Deduplicated by store_id.
    """
    logger.info("Transforming → silver.stores")

    stores_df = (
        pd.json_normalize(df["store"])
        .drop_duplicates(subset=["store_id"], keep="last")
        .reset_index(drop=True)
    )

    logger.info(f"silver.stores → {len(stores_df)} rows")
    return stores_df


# ── 5. silver.billing_counters  ──────────────────────────────
def transform_counters(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten billing_counter JSON → silver.billing_counters.
    Deduplicated by counter_id.
    """
    logger.info("Transforming → silver.billing_counters")

    counters_df = (
        pd.json_normalize(df["billing_counter"])
        .drop_duplicates(subset=["counter_id"], keep="last")
        .reset_index(drop=True)
    )

    logger.info(f"silver.billing_counters → {len(counters_df)} rows")
    return counters_df


# ── 6. silver.payments  ──────────────────────────────────────
def transform_payments(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten payment JSON → silver.payments.
    transaction_id is prepended so every row is traceable to its transaction.
    One row per transaction.
    """
    logger.info("Transforming → silver.payments")

    payments_df = pd.json_normalize(df["payment"]).copy()
    payments_df.insert(0, "transaction_id", df["transaction_id"].values)

    logger.info(f"silver.payments → {len(payments_df)} rows")
    return payments_df


# ── 7. silver.returns  ───────────────────────────────────────
def transform_returns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten return_info JSON → silver.returns.
    Only transactions that contain actual return data are included.
    transaction_id is prepended for traceability.
    """
    logger.info("Transforming → silver.returns")

    # Keep only rows where return_info is a non-empty dict
    mask = df["return_info"].apply(lambda x: isinstance(x, dict) and bool(x))
    returns_source = df[mask].reset_index(drop=True)

    if returns_source.empty:
        logger.info("silver.returns → 0 rows (no return records in this batch)")
        return pd.DataFrame()

    returns_df = pd.json_normalize(returns_source["return_info"]).copy()
    returns_df.insert(0, "transaction_id", returns_source["transaction_id"].values)

    logger.info(f"silver.returns → {len(returns_df)} rows")
    return returns_df


# ── 8. silver.stock_movements  ───────────────────────────────
def transform_stock(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive inventory movements from sold items → silver.stock_movements.
    quantity_change is NEGATIVE (stock out on each sale).
    One row per line item per transaction.
    """
    logger.info("Transforming → silver.stock_movements")

    rows = []
    for _, row in df.iterrows():
        store_id = (
            row["store"].get("store_id")
            if isinstance(row.get("store"), dict) else None
        )
        items = row.get("items") or []
        for item in items:
            if not isinstance(item, dict):
                continue
            rows.append({
                "product_id":         item.get("product_id"),
                "store_id":           store_id,
                "transaction_id":     row["transaction_id"],
                "movement_type":      "SALE",
                "quantity_change":    -(item.get("quantity", 0)),
                "movement_timestamp": row["transaction_timestamp"],
                "sale_date":          row["transaction_date"],
                "reference_type":     "SALE",
                "reference_id":       row["transaction_id"],
            })

    stock_df = pd.DataFrame(rows)
    logger.info(f"silver.stock_movements → {len(stock_df)} rows")
    return stock_df


# ============================================================
# LOAD
# ============================================================
def truncate_silver_tables(engine: Engine) -> None:
    """Truncate all silver tables. Used only for full load mode."""
    logger.info("Full load — truncating all silver tables")
    with engine.begin() as conn:
        for table in SILVER_TABLES:
            conn.execute(
                text(f"TRUNCATE TABLE silver.{table} RESTART IDENTITY CASCADE")
            )
            logger.info(f"Truncated silver.{table}")


def load_to_silver(
    df:        pd.DataFrame,
    engine:    Engine,
    table:     str,
    load_type: str = "incremental",
) -> None:
    """
    Append a transformed DataFrame into a silver schema table.

    Args:
        df:        Transformed DataFrame to load.
        engine:    Destination DB engine.
        table:     Target table name (without schema prefix).
        load_type: Used for logging context only —
                   truncation is handled upstream in main().
    """
    if df is None or df.empty:
        logger.warning(f"Skipping silver.{table} — DataFrame is empty")
        return

    # Drop any accidentally duplicated columns before loading
    df = df.loc[:, ~df.columns.duplicated()].reset_index(drop=True)

    logger.info(f"Loading silver.{table} | {len(df)} rows | mode={load_type}")
    df.to_sql(
        name=table,
        con=engine,
        schema="silver",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=500,
    )
    logger.info(f"Loaded {len(df)} rows → silver.{table}")


# ============================================================
# MAIN
# ============================================================
def main(
    source_db:  DBConfig,
    dest_db:    DBConfig,
    start_date: Optional[str] = None,
    end_date:   Optional[str] = None,
    load_type:  str = "incremental",
) -> None:
    """
    End-to-end Bronze → Silver pipeline.

    Args:
        source_db:  Connection config for the bronze (source) database.
        dest_db:    Connection config for the silver (destination) database.
        start_date: Filter lower bound (YYYY-MM-DD). None = no filter.
        end_date:   Filter upper bound (YYYY-MM-DD). None = no filter.
        load_type:  "full"        → truncate silver tables then reload.
                    "incremental" → append only new/changed records.
    """
    logger.info("=" * 60)
    logger.info("Bronze → Silver Pipeline  START")
    logger.info(f"load_type  : {load_type}")
    logger.info(f"start_date : {start_date or 'ALL'}")
    logger.info(f"end_date   : {end_date   or 'ALL'}")
    logger.info("=" * 60)

    # ── connections ──────────────────────────────────────────
    source_engine = get_engine(source_db)
    dest_engine   = get_engine(dest_db)

    # ── full load: clear destination first ───────────────────
    if load_type == "full":
        truncate_silver_tables(dest_engine)

    # ── extract ──────────────────────────────────────────────
    raw_df = extract_bronze(source_engine, start_date, end_date)

    if raw_df.empty:
        logger.warning("No records found for the given range — pipeline exiting")
        return

    # ── parse JSON strings → Python objects ──────────────────
    parsed_df = parse_json_columns(raw_df)

    # ── transform (all 8 tables) ─────────────────────────────
    sales_df       = transform_sales(parsed_df)
    sales_items_df = transform_sales_items(parsed_df)
    customers_df   = transform_customers(parsed_df)
    stores_df      = transform_stores(parsed_df)
    counters_df    = transform_counters(parsed_df)
    payments_df    = transform_payments(parsed_df)
    returns_df     = transform_returns(parsed_df)
    stock_df       = transform_stock(parsed_df)

    # ── load (all 8 tables) ───────────────────────────────────
    load_to_silver(sales_df,       dest_engine, "sales",            load_type)
    load_to_silver(sales_items_df, dest_engine, "sales_items",      load_type)
    load_to_silver(customers_df,   dest_engine, "customers",        load_type)
    load_to_silver(stores_df,      dest_engine, "stores",           load_type)
    load_to_silver(counters_df,    dest_engine, "billing_counters", load_type)
    load_to_silver(payments_df,    dest_engine, "payments",         load_type)
    load_to_silver(returns_df,     dest_engine, "returns",          load_type)
    load_to_silver(stock_df,       dest_engine, "stock_movements",  load_type)

    logger.info("=" * 60)
    logger.info("Bronze → Silver Pipeline  COMPLETE")
    logger.info("=" * 60)


: 

In [ ]:


# ============================================================
# ENTRY POINT
# ============================================================
if __name__ == "__main__":

    source_db = DBConfig(
        host     = "localhost",
        db       = "retail_db",
        user     = "postgres",
        password = "password",
        port     = 5432,
    )

    dest_db = DBConfig(
        host     = "localhost",
        db       = "retail_db",
        user     = "postgres",
        password = "password",
        port     = 5432,
    )

    main(
        source_db  = source_db,
        dest_db    = dest_db,
        start_date = "2024-01-01",
        end_date   = "2024-01-31",
        load_type  = "incremental",
    )